# Module 11 Applied Lab — Weather API Observability

**Session anchor:** 90 minutes (Day 2)
**Stack:** FastAPI · prometheus_client · httpx
**Goal:** Instrument a single-endpoint weather service end-to-end

---

| Beat | Topic | Time |
|------|-------|------|
| 1 | Setup + orientation: weather-API skeleton | 10 min |
| 2 | Three metric declarations | 15 min |
| 3 | Mounting `/metrics` | 8 min |
| 4 | Request-ID middleware | 10 min |
| 5 | Structured-logging middleware | 10 min |
| 6 | Metrics middleware + label-cardinality pitfall | 15 min |
| 7 | Reading `/metrics` from a Python client | 10 min |

## Beat 1 — Setup + Orientation (10 min)

### What we're building

A single-endpoint FastAPI service that answers `GET /weather?city=...` with a forecast string
pulled from an in-memory dict. We will layer **three Prometheus metrics**, a **request-ID header**,
and **structured JSON logs** on top of it.

### File layout

```
Module 11 Applied Lab/
├── app.py              # FastAPI application (built incrementally through the beats)
├── requirements.txt    # Python dependencies
├── read_metrics.py     # client that parses /metrics output (Beat 7)
└── tests/
    └── test_metrics.py # pytest suite (written live in session)
```

### Why this domain?

The four-service stack from Module 10A is **not here**.
The instrumentation patterns — counter, histogram, gauge, middleware order, label cardinality — are identical.
You must *apply* the patterns, not copy the code.

In [44]:
# ── Pin working directory to this notebook's folder (VS Code Jupyter fix) ─────
# VS Code Jupyter kernels start in the workspace root, not the notebook folder.
# %%writefile and subprocess.Popen inherit CWD, so we must set it explicitly.
try:
    import os as _os
    _os.chdir(_os.path.dirname(__vsc_ipynb_file__))         # __vsc_ipynb_file__ = VS Code notebook path
except NameError:
    pass                                                    # Classic Jupyter: CWD is already correct
# ──────────────────────────────────────────────────────────────────────────────

import subprocess                                                # runs shell commands from Python
import sys                                                       # access to interpreter info
import os                                                        # OS-level file/directory operations
import time                                                      # wall-clock timing utilities
import json                                                      # JSON serialization

LAB_DIR = os.getcwd()                                           # notebook working dir = lab root

os.makedirs("tests", exist_ok=True)                             # create tests/ directory (idempotent)

# ── Verify and install required packages ──────────────────────────────────────
required = ["fastapi", "uvicorn", "prometheus_client", "httpx"] # packages needed for this lab

print("Checking installed packages…")
for pkg in required:                                            # iterate over each required package
    result = subprocess.run(                                    # attempt to locate the package
        [sys.executable, "-m", "pip", "show", pkg],            # pip show exits 0 if installed
        capture_output=True,                                    # suppress output during check
    )
    status = "✓" if result.returncode == 0 else "✗ MISSING"   # ✓ found  ✗ needs install
    print(f"  {status}  {pkg}")

print("\nInstalling / upgrading packages…")
subprocess.run(                                                 # install all dependencies at once
    [sys.executable, "-m", "pip", "install",
     "fastapi==0.111.0",
     "uvicorn[standard]==0.30.1",
     "prometheus_client==0.20.0",
     "httpx==0.27.0",
     "--quiet"],                                                # suppress verbose pip output
    check=True,                                                 # raise error if install fails
)
print("All packages ready.")                                    # confirm successful install

Checking installed packages…
  ✓  fastapi
  ✓  uvicorn
  ✓  prometheus_client
  ✓  httpx

Installing / upgrading packages…
All packages ready.


In [45]:
%%writefile requirements.txt
fastapi==0.111.0            # ASGI web framework with automatic OpenAPI docs
uvicorn[standard]==0.30.1   # ASGI server; [standard] adds websocket + watchfiles reload

Overwriting requirements.txt


In [46]:
%%writefile app.py
"""Weather API — Beat 1 skeleton: route only, no instrumentation."""
from fastapi import FastAPI                                      # FastAPI application class
from fastapi.responses import JSONResponse                       # helper for explicit JSON responses

app = FastAPI(title="Weather API")                              # create the ASGI application instance

# ── In-memory data store ──────────────────────────────────────────────────────
FORECASTS = {                                                    # dict: city name → forecast string
    "Boston":    "Partly cloudy, 18 °C",                        # northeast US city
    "Dubai":     "Sunny, 41 °C",                                # UAE desert city
    "London":    "Overcast, 12 °C",                             # UK capital
    "New York":  "Thunderstorms, 22 °C",                        # northeast US metro
    "Tokyo":     "Clear, 26 °C",                                # Japanese capital
}

# ── Route ─────────────────────────────────────────────────────────────────────
@app.get("/weather")                                             # registers a GET handler at /weather
def get_weather(city: str):                                      # city is a required query param (?city=…)
    forecast = FORECASTS.get(city)                              # look up city; None if not found
    if forecast is None:                                        # city absent from the data store
        return JSONResponse(                                    # return a 404 JSON error body
            status_code=404,
            content={"error": f"No forecast for '{city}'"},
        )
    return {"city": city, "forecast": forecast}                 # happy path: city + forecast as JSON

Overwriting app.py


In [47]:
import subprocess, sys, time, httpx, os

# ── Derive lab_dir so this cell works even if run before the prerequisites cell ─
import os as _os
try:
    lab_dir = _os.path.dirname(__vsc_ipynb_file__)          # VS Code: absolute path to notebook folder
    _os.chdir(lab_dir)                                      # ensure CWD = notebook directory
except NameError:
    lab_dir = _os.getcwd()                                  # Classic Jupyter: CWD already correct


# ── Kill any running server from a previous cell ──────────────────────────────
subprocess.run(["pkill", "-f", "uvicorn app:app"],              # send SIGTERM to any uvicorn process
               capture_output=True)
time.sleep(1)                                                   # brief pause to free port 8001

# ── Start uvicorn in the background ──────────────────────────────────────────
server = subprocess.Popen(                                      # Popen = non-blocking process launch
    [sys.executable, "-m", "uvicorn", "app:app",               # run uvicorn serving our app module
     "--port", "8001",                                          # bind to port 8001
     "--log-level", "warning"],                                 # suppress info-level boot chatter
    stdout=subprocess.PIPE,                                     # capture stdout
    stderr=subprocess.STDOUT,                                   # merge stderr → stdout stream
    cwd=lab_dir,                                       # run uvicorn from the notebook directory
)
time.sleep(2)                                                   # wait for uvicorn to finish starting

# ── Smoke test ────────────────────────────────────────────────────────────────
resp = httpx.get("http://localhost:8001/weather?city=Boston",   # hit the weather endpoint
                  timeout=5)                                    # fail fast if server is down
print(f"Status : {resp.status_code}")                          # should print 200
print(f"Body   : {resp.json()}")                               # should print city + forecast dict

resp404 = httpx.get("http://localhost:8001/weather?city=Atlantis",  # unknown city → 404
                     timeout=5)
print(f"\n404 test — Status : {resp404.status_code}")          # should print 404
print(f"404 test — Body   : {resp404.json()}")                 # should print error dict

Status : 200
Body   : {'city': 'Boston', 'forecast': 'Partly cloudy, 18 °C'}

404 test — Status : 404
404 test — Body   : {'error': "No forecast for 'Atlantis'"}


## Beat 2 — Three Metric Declarations (15 min)

### Metric type selection

| Signal | Type | Why |
|--------|------|-----|
| Requests completed | **Counter** | Only goes up; use `rate()` in PromQL for req/s |
| Request duration | **Histogram** | Captures p50 / p95 / p99 from bucketed samples |
| Requests in-flight | **Gauge** | Goes up *and* down; reflects concurrent load right now |

### The module-scope rule

Metrics must be declared **once**, at import time, outside any function.
Declaring inside a request handler re-registers on every call → `ValueError: Duplicated timeseries`.

### Labels

Labels multiply a metric into separate time series.
Start with one or two labels whose values form a **bounded, known set**.
We will violate this rule in Beat 6 to see the explosion live.

In [48]:
%%writefile requirements.txt
fastapi==0.111.0            # ASGI web framework with automatic OpenAPI docs
uvicorn[standard]==0.30.1   # ASGI server; [standard] adds websocket + watchfiles reload
prometheus_client==0.20.0   # Prometheus instrumentation library for Python

Overwriting requirements.txt


In [49]:
%%writefile app.py
"""Weather API — Beat 2: metric declarations added at module scope."""
from fastapi import FastAPI                                      # FastAPI application class
from fastapi.responses import JSONResponse                       # helper for explicit JSON responses
from prometheus_client import Counter, Histogram, Gauge          # the three metric type classes we need

app = FastAPI(title="Weather API")                              # create the ASGI application instance

# ── In-memory data store ──────────────────────────────────────────────────────
FORECASTS = {                                                    # dict: city name → forecast string
    "Boston":    "Partly cloudy, 18 °C",
    "Dubai":     "Sunny, 41 °C",
    "London":    "Overcast, 12 °C",
    "New York":  "Thunderstorms, 22 °C",
    "Tokyo":     "Clear, 26 °C",
}

# ── Metric declarations (module scope — created once at import time) ──────────
REQUESTS_TOTAL = Counter(                                        # Counter: monotonically increasing
    "requests_total",                                            # metric name (appears in /metrics)
    "Total HTTP requests",                                       # description → # HELP line
    ["path", "status"],                                          # label names (bounded: paths + codes)
)

REQUEST_LATENCY = Histogram(                                     # Histogram: bucketed duration samples
    "request_latency_seconds",                                   # metric name
    "HTTP request latency in seconds",                           # description
    ["path"],                                                    # one label: path (not status — too many buckets)
)

INFLIGHT_REQUESTS = Gauge(                                       # Gauge: value goes up and down
    "inflight_requests",                                         # metric name
    "Number of requests currently in-flight",                    # description
)                                                                # no labels — one global count is enough

# ── Route ─────────────────────────────────────────────────────────────────────
@app.get("/weather")                                             # GET /weather handler
def get_weather(city: str):                                      # city query parameter
    forecast = FORECASTS.get(city)
    if forecast is None:
        return JSONResponse(status_code=404, content={"error": f"No forecast for '{city}'"})
    return {"city": city, "forecast": forecast}

Overwriting app.py


In [50]:
# ── Demonstrate the duplicate-registration error ──────────────────────────────
# Run this cell to see what happens when you declare a metric inside a function.
# Then read the error, and notice we already declared REQUESTS_TOTAL at module scope above.

from prometheus_client import Counter

def bad_handler():
    c = Counter("requests_total", "oops", ["path", "status"])  # re-registering an existing name
    c.labels(path="/weather", status="200").inc()

try:
    bad_handler()                                               # first call registers
    bad_handler()                                               # second call collides → ValueError
except ValueError as e:
    print(f"ValueError caught: {e}")                           # shows: Duplicated timeseries in CollectorRegistry
print("\n→ Always declare metrics at module scope, not inside request handlers.")

ValueError caught: Duplicated timeseries in CollectorRegistry: {'requests_created', 'requests_total', 'requests'}

→ Always declare metrics at module scope, not inside request handlers.


## Beat 3 — Mounting `/metrics` (8 min)

`prometheus_client.make_asgi_app()` returns a lightweight ASGI application that reads the
default Prometheus registry and serialises it as OpenMetrics text.

`app.mount("/metrics", metrics_app)` splices it into the FastAPI router: any `GET /metrics`
request is handled by the exporter, not your route functions.

### What you'll see *before* any middleware exists

```
# HELP requests_total Total HTTP requests
# TYPE requests_total counter
# HELP request_latency_seconds HTTP request latency in seconds
# TYPE request_latency_seconds histogram
# HELP inflight_requests Number of requests currently in-flight
# TYPE inflight_requests gauge
inflight_requests 0.0
```

The `# HELP` and `# TYPE` comment lines are the **contract surface**: they confirm the metrics
are registered. No sample lines for the counter and histogram yet — no middleware is
incrementing them.

In [51]:
%%writefile app.py
"""Weather API — Beat 3: /metrics endpoint mounted."""
from fastapi import FastAPI                                      # FastAPI application class
from fastapi.responses import JSONResponse                       # helper for explicit JSON responses
from prometheus_client import (                                  # import metric types + ASGI factory
    Counter,                                                     # monotonically increasing counter
    Histogram,                                                   # bucketed duration distribution
    Gauge,                                                       # point-in-time mutable value
    make_asgi_app,                                               # factory: Prometheus registry → ASGI app
)

app = FastAPI(title="Weather API")                              # create the ASGI application instance

FORECASTS = {
    "Boston":    "Partly cloudy, 18 °C",
    "Dubai":     "Sunny, 41 °C",
    "London":    "Overcast, 12 °C",
    "New York":  "Thunderstorms, 22 °C",
    "Tokyo":     "Clear, 26 °C",
}

REQUESTS_TOTAL = Counter(
    "requests_total", "Total HTTP requests", ["path", "status"],
)
REQUEST_LATENCY = Histogram(
    "request_latency_seconds", "HTTP request latency in seconds", ["path"],
)
INFLIGHT_REQUESTS = Gauge(
    "inflight_requests", "Number of requests currently in-flight",
)

# ── Mount /metrics endpoint ───────────────────────────────────────────────────
metrics_app = make_asgi_app()                                    # build Prometheus ASGI sub-application
app.mount("/metrics", metrics_app)                               # attach exporter at /metrics path

# ── Route ─────────────────────────────────────────────────────────────────────
@app.get("/weather")
def get_weather(city: str):
    forecast = FORECASTS.get(city)
    if forecast is None:
        return JSONResponse(status_code=404, content={"error": f"No forecast for '{city}'"})
    return {"city": city, "forecast": forecast}

Overwriting app.py


In [52]:
import subprocess, sys, time, httpx

# ── Derive lab_dir so this cell works even if run before the prerequisites cell ─
import os as _os
try:
    lab_dir = _os.path.dirname(__vsc_ipynb_file__)          # VS Code: absolute path to notebook folder
    _os.chdir(lab_dir)                                      # ensure CWD = notebook directory
except NameError:
    lab_dir = _os.getcwd()                                  # Classic Jupyter: CWD already correct


subprocess.run(["pkill", "-f", "uvicorn app:app"], capture_output=True)  # stop previous server
time.sleep(1)

server = subprocess.Popen(                                      # start fresh server process
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8001",
     "--log-level", "warning"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=lab_dir,                                       # run uvicorn from the notebook directory
)
time.sleep(2)                                                   # wait for startup

# ── Read /metrics output ──────────────────────────────────────────────────────
# follow_redirects=True: Starlette redirects /metrics → /metrics/ (307); httpx
# 0.20+ does NOT follow redirects by default, so we must opt in explicitly.
metrics_text = httpx.get("http://localhost:8001/metrics",
                          timeout=5, follow_redirects=True).text
# Print just the # HELP / # TYPE lines + the inflight_requests sample
for line in metrics_text.splitlines():                          # iterate over each output line
    if line.startswith("#") or line.startswith("inflight"):     # keep HELP/TYPE lines + gauge sample
        print(line)

# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
# HELP python_info Python platform information
# TYPE python_info gauge
# HELP process_virtual_memory_bytes Virtual memory size in bytes.
# TYPE process_virtual_memory_bytes gauge
# HELP process_resident_memory_bytes Resident memory size in bytes.
# TYPE process_resident_memory_bytes gauge
# HELP process_start_time_seconds Start time of the process since unix epoch in seconds.
# TYPE process_start_time_seconds gauge
# HELP process_cpu_seconds_total Total user and system CPU time spent in seconds.
# TYPE process_cpu_seconds_total counter
# HELP process_open_fds Number of open file descriptors.
# TYPE process_open_fd

## Beat 4 — Request-ID Middleware (10 min)

Every request gets a unique `request_id` that flows through logs, metrics, and the response
header. We use a `ContextVar` — Python's async-safe per-task storage — so the ID is accessible
anywhere in the call stack without thread-local hacks.

### Middleware ordering

FastAPI processes middleware in **reverse registration order** (last registered = outermost).
We will register three middleware functions in this order:

1. `logging_middleware` (outermost — wraps everything)
2. `metrics_middleware`
3. `request_id_middleware` (innermost — sets the ID before any other layer reads it)

> **Rule:** Register the middleware that *produces* a value before the middleware that *consumes* it.

In [53]:
%%writefile app.py
"""Weather API — Beat 4: request-ID ContextVar + middleware added."""
from fastapi import FastAPI, Request                             # Request gives us headers, url, etc.
from fastapi.responses import JSONResponse
from prometheus_client import Counter, Histogram, Gauge, make_asgi_app
from contextvars import ContextVar                               # async-safe per-task storage
import uuid                                                      # generates universally unique IDs

app = FastAPI(title="Weather API")

FORECASTS = {
    "Boston":    "Partly cloudy, 18 °C",
    "Dubai":     "Sunny, 41 °C",
    "London":    "Overcast, 12 °C",
    "New York":  "Thunderstorms, 22 °C",
    "Tokyo":     "Clear, 26 °C",
}

REQUESTS_TOTAL = Counter("requests_total", "Total HTTP requests", ["path", "status"])
REQUEST_LATENCY = Histogram("request_latency_seconds", "HTTP request latency in seconds", ["path"])
INFLIGHT_REQUESTS = Gauge("inflight_requests", "Number of requests currently in-flight")

# ── Request-ID context storage ────────────────────────────────────────────────
request_id_var: ContextVar[str] = ContextVar(                   # declare module-level ContextVar
    "request_id", default=""                                    # default empty string when no request
)

metrics_app = make_asgi_app()
app.mount("/metrics", metrics_app)

# ── Middleware ────────────────────────────────────────────────────────────────
@app.middleware("http")                                          # registers an HTTP middleware function
async def request_id_middleware(request: Request, call_next):   # called for every incoming request
    rid = uuid.uuid4().hex                                      # generate 32-char hex unique ID
    request_id_var.set(rid)                                     # store in ContextVar for this async task
    response = await call_next(request)                         # forward request to next layer
    response.headers["X-Request-ID"] = rid                      # attach ID to outgoing response header
    return response                                             # return completed response to client

# ── Route ─────────────────────────────────────────────────────────────────────
@app.get("/weather")
def get_weather(city: str):
    forecast = FORECASTS.get(city)
    if forecast is None:
        return JSONResponse(status_code=404, content={"error": f"No forecast for '{city}'"})
    return {"city": city, "forecast": forecast}

Overwriting app.py


In [54]:
import subprocess, sys, time, httpx

# ── Derive lab_dir so this cell works even if run before the prerequisites cell ─
import os as _os
try:
    lab_dir = _os.path.dirname(__vsc_ipynb_file__)          # VS Code: absolute path to notebook folder
    _os.chdir(lab_dir)                                      # ensure CWD = notebook directory
except NameError:
    lab_dir = _os.getcwd()                                  # Classic Jupyter: CWD already correct


subprocess.run(["pkill", "-f", "uvicorn app:app"], capture_output=True)
time.sleep(1)
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8001", "--log-level", "warning"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=lab_dir,                                       # run uvicorn from the notebook directory
)
time.sleep(2)

# ── Show the X-Request-ID response header ─────────────────────────────────────
resp = httpx.get("http://localhost:8001/weather?city=Tokyo", timeout=5)  # make a request
print("Response headers:")
for name, value in resp.headers.items():                        # iterate over all response headers
    if "request-id" in name.lower() or "content" in name.lower():  # filter to relevant headers
        print(f"  {name}: {value}")

print(f"\nRequest ID: {resp.headers.get('x-request-id')}")    # pull the X-Request-ID header value
print("\nMake two more requests — each gets a unique ID:")
for _ in range(2):
    r = httpx.get("http://localhost:8001/weather?city=Dubai", timeout=5)
    print(f"  {r.headers.get('x-request-id')}")                # each hex string should be different

Response headers:
  content-length: 43
  content-type: application/json
  x-request-id: 396139ef9fc1408fbcd379e7d98ccc9e

Request ID: 396139ef9fc1408fbcd379e7d98ccc9e

Make two more requests — each gets a unique ID:
  9c875210de224c04b729f7609b8d600b
  fea850f87a2e4d4099b09e5c715c827e


## Beat 5 — Structured-Logging Middleware (10 min)

One JSON line per request. Every field is machine-readable: ingest it into any log aggregator
(Loki, CloudWatch, Datadog) and `jq`-filter it on the command line.

### JSON log line shape

```json
{
  "ts": "2026-06-26T14:03:11",
  "level": "INFO",
  "request_id": "a3f8c1e9d2b74056…",
  "path": "/weather",
  "status": 200,
  "latency_ms": 1.42,
  "msg": "request"
}
```

### Key design choices

| Choice | Reason |
|--------|--------|
| `logging.Formatter` subclass | Plugs into standard Python logging — no new dependency |
| `ContextVar.get()` inside `format()` | Pulls the current request's ID without passing it explicitly |
| `extra={}` in `logger.info()` | Injects per-request fields into the `LogRecord` without polluting the message |

In [55]:
%%writefile app.py
"""Weather API — Beat 5: structured JSON logging middleware added."""
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from prometheus_client import Counter, Histogram, Gauge, make_asgi_app
from contextvars import ContextVar
import uuid                                                      # generates universally unique IDs
import logging                                                   # Python standard logging framework
import json                                                      # JSON serialization
import time                                                      # wall-clock timing

app = FastAPI(title="Weather API")

FORECASTS = {
    "Boston":    "Partly cloudy, 18 °C",
    "Dubai":     "Sunny, 41 °C",
    "London":    "Overcast, 12 °C",
    "New York":  "Thunderstorms, 22 °C",
    "Tokyo":     "Clear, 26 °C",
}

REQUESTS_TOTAL = Counter("requests_total", "Total HTTP requests", ["path", "status"])
REQUEST_LATENCY = Histogram("request_latency_seconds", "HTTP request latency in seconds", ["path"])
INFLIGHT_REQUESTS = Gauge("inflight_requests", "Number of requests currently in-flight")

request_id_var: ContextVar[str] = ContextVar("request_id", default="")

# ── Structured JSON log formatter ─────────────────────────────────────────────
class JSONFormatter(logging.Formatter):                          # subclass standard Formatter
    def format(self, record: logging.LogRecord) -> str:         # override to emit JSON instead of text
        return json.dumps({                                      # serialize all fields to one JSON line
            "ts":         self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),  # ISO 8601 timestamp
            "level":      record.levelname,                     # DEBUG / INFO / WARNING / ERROR
            "request_id": request_id_var.get(),                 # pull ID from ContextVar (async-safe)
            "path":       getattr(record, "path", ""),          # URL path — injected via extra={}
            "status":     getattr(record, "status", ""),        # HTTP status — injected via extra={}
            "latency_ms": getattr(record, "latency_ms", ""),    # latency in ms — injected via extra={}
            "msg":        record.getMessage(),                  # the string passed to logger.info(…)
        })

handler = logging.StreamHandler()                               # writes log records to stderr
handler.setFormatter(JSONFormatter())                           # attach our JSON formatter to it
logger = logging.getLogger("weather")                           # named logger for this application
logger.addHandler(handler)                                      # wire the handler to the logger
logger.setLevel(logging.INFO)                                   # INFO and above; suppress DEBUG noise
logger.propagate = False                                        # prevent duplicate lines in root logger

metrics_app = make_asgi_app()
app.mount("/metrics", metrics_app)

# ── Middleware ────────────────────────────────────────────────────────────────
@app.middleware("http")
async def request_id_middleware(request: Request, call_next):
    rid = uuid.uuid4().hex
    request_id_var.set(rid)
    response = await call_next(request)
    response.headers["X-Request-ID"] = rid
    return response

@app.middleware("http")                                          # second middleware registration
async def logging_middleware(request: Request, call_next):      # emits one JSON log line per request
    start = time.perf_counter()                                 # capture start time for latency calc
    response = await call_next(request)                         # pass request through inner layers
    latency_ms = (time.perf_counter() - start) * 1000          # convert elapsed seconds → ms
    logger.info(                                                # emit at INFO level
        "request",                                              # the msg field in the JSON line
        extra={                                                 # extra dict injects fields into LogRecord
            "path":       request.url.path,                    # URL path (e.g. /weather)
            "status":     response.status_code,                # HTTP status code integer
            "latency_ms": round(latency_ms, 2),                # latency rounded to 2 decimal places
        },
    )
    return response                                             # pass response back up the chain

# ── Route ─────────────────────────────────────────────────────────────────────
@app.get("/weather")
def get_weather(city: str):
    forecast = FORECASTS.get(city)
    if forecast is None:
        return JSONResponse(status_code=404, content={"error": f"No forecast for '{city}'"})
    return {"city": city, "forecast": forecast}

Overwriting app.py


In [56]:
import subprocess, sys, time, httpx, json

# ── Derive lab_dir so this cell works even if run before the prerequisites cell ─
import os as _os
try:
    lab_dir = _os.path.dirname(__vsc_ipynb_file__)          # VS Code: absolute path to notebook folder
    _os.chdir(lab_dir)                                      # ensure CWD = notebook directory
except NameError:
    lab_dir = _os.getcwd()                                  # Classic Jupyter: CWD already correct


subprocess.run(["pkill", "-f", "uvicorn app:app"], capture_output=True)
time.sleep(1)
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8001", "--log-level", "warning"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=lab_dir,                                       # run uvicorn from the notebook directory
)
time.sleep(2)

# ── Make a request and read the log line from server stderr ──────────────────
resp = httpx.get("http://localhost:8001/weather?city=London", timeout=5)  # trigger logging
time.sleep(0.3)                                                 # give server time to flush log line

# Read one line of output from the server process stdout (merged stderr)
raw_line = server.stdout.readline().decode("utf-8", errors="replace").strip()  # read server output

try:
    log_entry = json.loads(raw_line)                            # parse JSON log line
    print("Parsed log entry:")
    for k, v in log_entry.items():                             # pretty-print each field
        print(f"  {k:12s}: {v}")
except json.JSONDecodeError:
    print(f"Raw output: {raw_line}")                           # fallback if uvicorn emitted non-JSON
    print("(Tip: the JSON line may be on a subsequent read — run the cell again)")

Parsed log entry:
  ts          : 2026-06-26T14:52:01
  level       : INFO
  request_id  : 
  path        : /weather
  status      : 200
  latency_ms  : 3.85
  msg         : request


## Beat 6 — Metrics Middleware + Label-Cardinality Pitfall (15 min)

The metrics middleware:
1. **Increments** the in-flight gauge *before* `await call_next`
2. **Times** the request with `time.perf_counter()`
3. **Decrements** the gauge after the response returns
4. **Increments** `requests_total` with `path` and `status` labels
5. **Observes** `request_latency_seconds` with the measured duration

### The cardinality pitfall

After wiring the correct middleware we will **intentionally break it** by adding `city` (a query
parameter with unbounded values) as a label on `requests_total`. Five different cities → five
separate counter time series. In production with millions of cities the Prometheus server runs
out of memory. We will see the explosion in the `/metrics` output, then roll back.

In [57]:
%%writefile app.py
"""Weather API — Beat 6: metrics middleware added (full instrumented version)."""
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from prometheus_client import Counter, Histogram, Gauge, make_asgi_app
from contextvars import ContextVar
import uuid
import logging
import json
import time                                                      # perf_counter for sub-millisecond timing

app = FastAPI(title="Weather API")

FORECASTS = {
    "Boston":    "Partly cloudy, 18 °C",
    "Dubai":     "Sunny, 41 °C",
    "London":    "Overcast, 12 °C",
    "New York":  "Thunderstorms, 22 °C",
    "Tokyo":     "Clear, 26 °C",
}

REQUESTS_TOTAL = Counter("requests_total", "Total HTTP requests", ["path", "status"])
REQUEST_LATENCY = Histogram("request_latency_seconds", "HTTP request latency in seconds", ["path"])
INFLIGHT_REQUESTS = Gauge("inflight_requests", "Number of requests currently in-flight")

request_id_var: ContextVar[str] = ContextVar("request_id", default="")

class JSONFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        return json.dumps({
            "ts":         self.formatTime(record, "%Y-%m-%dT%H:%M:%S"),
            "level":      record.levelname,
            "request_id": request_id_var.get(),
            "path":       getattr(record, "path", ""),
            "status":     getattr(record, "status", ""),
            "latency_ms": getattr(record, "latency_ms", ""),
            "msg":        record.getMessage(),
        })

handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())
logger = logging.getLogger("weather")
logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

metrics_app = make_asgi_app()
app.mount("/metrics", metrics_app)

# ── Middleware (registered in reverse — last registered = outermost) ──────────
@app.middleware("http")                                          # innermost: sets request_id first
async def request_id_middleware(request: Request, call_next):
    rid = uuid.uuid4().hex                                      # 32-char hex unique request ID
    request_id_var.set(rid)                                     # bind ID to this async task
    response = await call_next(request)
    response.headers["X-Request-ID"] = rid
    return response

@app.middleware("http")                                         # middle: times + increments metrics
async def metrics_middleware(request: Request, call_next):      # wraps the route handler
    INFLIGHT_REQUESTS.inc()                                     # +1 before processing starts
    start = time.perf_counter()                                 # record high-resolution start time
    response = await call_next(request)                         # run route handler
    latency = time.perf_counter() - start                      # elapsed seconds (float)
    INFLIGHT_REQUESTS.dec()                                     # -1 after response is ready
    path   = request.url.path                                   # e.g. "/weather"
    status = str(response.status_code)                          # convert int → string for label
    REQUESTS_TOTAL.labels(path=path, status=status).inc()       # increment counter for this path+status combo
    REQUEST_LATENCY.labels(path=path).observe(latency)          # record duration in histogram buckets
    return response

@app.middleware("http")                                         # outermost: logs after full round-trip
async def logging_middleware(request: Request, call_next):
    start = time.perf_counter()
    response = await call_next(request)
    latency_ms = (time.perf_counter() - start) * 1000
    logger.info("request", extra={
        "path":       request.url.path,
        "status":     response.status_code,
        "latency_ms": round(latency_ms, 2),
    })
    return response

# ── Route ─────────────────────────────────────────────────────────────────────
@app.get("/weather")                                             # GET /weather?city=… handler
def get_weather(city: str):                                      # city is a required query parameter
    forecast = FORECASTS.get(city)                              # look up city in FORECASTS dict
    if forecast is None:                                        # city not in our data store
        return JSONResponse(                                    # return 404 JSON error
            status_code=404,
            content={"error": f"No forecast for '{city}'"},
        )
    return {"city": city, "forecast": forecast}                 # success: return city + forecast

Overwriting app.py


In [58]:
import subprocess, sys, time, httpx

# ── Derive lab_dir so this cell works even if run before the prerequisites cell ─
import os as _os
try:
    lab_dir = _os.path.dirname(__vsc_ipynb_file__)          # VS Code: absolute path to notebook folder
    _os.chdir(lab_dir)                                      # ensure CWD = notebook directory
except NameError:
    lab_dir = _os.getcwd()                                  # Classic Jupyter: CWD already correct


subprocess.run(["pkill", "-f", "uvicorn app:app"], capture_output=True)
time.sleep(1)
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--port", "8001", "--log-level", "warning"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=lab_dir,                                       # run uvicorn from the notebook directory
)
time.sleep(2)

# ── Hit /weather 5 times ──────────────────────────────────────────────────────
cities = ["Boston", "Dubai", "London", "New York", "Tokyo"]
for city in cities:                                             # make one request per city
    r = httpx.get(f"http://localhost:8001/weather?city={city.replace(' ', '%20')}",
                  timeout=5)
    print(f"{city:12s} → {r.status_code}")                     # confirm each returns 200

# ── Read /metrics and show requests_total counter ────────────────────────────
# follow_redirects=True: Starlette redirects /metrics → /metrics/ (307); httpx
# 0.20+ does NOT follow redirects by default, so we must opt in explicitly.
print("\n── /metrics sample ──")
metrics_text = httpx.get("http://localhost:8001/metrics", timeout=5,
                          follow_redirects=True).text
for line in metrics_text.splitlines():                          # scan every line of the exporter output
    if line.startswith("requests_total") and not line.startswith("#"):  # filter to counter samples
        print(line)                                             # each line: labels + current count
    if line.startswith("request_latency_seconds_count"):        # histogram count lines
        print(line)
    if line.startswith("inflight_requests") and not line.startswith("#"):
        print(line)                                             # in-flight gauge (should be 0 at rest)

Boston       → 200
Dubai        → 200
London       → 200
New York     → 200
Tokyo        → 200

── /metrics sample ──
requests_total{path="/weather",status="200"} 5.0
requests_total{path="/metrics",status="307"} 1.0
request_latency_seconds_count{path="/weather"} 5.0
request_latency_seconds_count{path="/metrics"} 1.0
inflight_requests 1.0


In [59]:
# ── Cardinality pitfall demo ──────────────────────────────────────────────────
# We temporarily redefine REQUESTS_TOTAL with a 'city' label to show the explosion.
# This runs against the live server's /metrics — we parse the output, not re-run the server.
#
# The pattern: each unique (path, status, city) triple becomes its own time series.
# With 195 countries and thousands of cities, Prometheus runs out of memory.

print("Current requests_total time series in /metrics (correct — 2 labels):")
# follow_redirects=True: Starlette redirects /metrics → /metrics/ (307); httpx
# 0.20+ does NOT follow redirects by default, so we must opt in explicitly.
metrics_text = httpx.get("http://localhost:8001/metrics", timeout=5,
                          follow_redirects=True).text
counter_lines = [l for l in metrics_text.splitlines()          # collect all counter sample lines
                 if l.startswith("requests_total{")]            # filter to just counter samples
for line in counter_lines:
    print(f"  {line}")                                         # each line is one time series

print(f"\nTotal time series for requests_total: {len(counter_lines)}")  # small: 1-2 series

pitfall_msg = (
    "\n" +
    "-" * 53 + "\n" +
    "If we added city as a label:\n" +
    "  REQUESTS_TOTAL.labels(path=path, status=status, city=city).inc()\n\n" +
    "After hitting 5 cities:\n" +
    '  requests_total{path="/weather",status="200",city="Boston"}   1.0\n' +
    '  requests_total{path="/weather",status="200",city="Dubai"}    1.0\n' +
    '  requests_total{path="/weather",status="200",city="London"}   1.0\n' +
    '  requests_total{path="/weather",status="200",city="New York"} 1.0\n' +
    '  requests_total{path="/weather",status="200",city="Tokyo"}    1.0\n\n' +
    "In production with a real geocoded weather API:\n" +
    "  -> 195+ countries x thousands of cities x N status codes\n" +
    "  -> millions of time series -> Prometheus OOM\n\n" +
    "Rule: only use labels with a bounded, known set of values.\n" +
    "-" * 53
)
print(pitfall_msg)                                             # display the cardinality warning

Current requests_total time series in /metrics (correct — 2 labels):
  requests_total{path="/weather",status="200"} 5.0
  requests_total{path="/metrics",status="307"} 2.0
  requests_total{path="/metrics/",status="200"} 1.0

Total time series for requests_total: 3

-----------------------------------------------------
If we added city as a label:
  REQUESTS_TOTAL.labels(path=path, status=status, city=city).inc()

After hitting 5 cities:
  requests_total{path="/weather",status="200",city="Boston"}   1.0
  requests_total{path="/weather",status="200",city="Dubai"}    1.0
  requests_total{path="/weather",status="200",city="London"}   1.0
  requests_total{path="/weather",status="200",city="New York"} 1.0
  requests_total{path="/weather",status="200",city="Tokyo"}    1.0

In production with a real geocoded weather API:
  -> 195+ countries x thousands of cities x N status codes
  -> millions of time series -> Prometheus OOM

Rule: only use labels with a bounded, known set of values.
----------

## Beat 7 — Reading `/metrics` from a Python Client (10 min)

`prometheus_client` ships a text parser that transforms the OpenMetrics output back into
Python objects. The same pattern powers the integration's `lib/metrics_reader.py`.

### What `text_string_to_metric_families` returns

A generator of `MetricFamily` objects, each with:
- `.name` — metric name (e.g. `requests_total`)
- `.samples` — list of `Sample(name, labels, value, timestamp, exemplar)`

We iterate these to extract the value for a specific label combination.

In [60]:
%%writefile read_metrics.py
"""Beat 7 — parse /metrics from a running weather service."""
import httpx                                                     # HTTP client for fetching /metrics
from prometheus_client.parser import text_string_to_metric_families  # OpenMetrics text parser

BASE_URL = "http://localhost:8001"                              # weather service base URL

def fetch_metric(name: str) -> list:                            # returns list of Sample objects
    """Fetch /metrics and extract all samples for a given metric name."""
    # follow_redirects=True: Starlette redirects /metrics → /metrics/ (307); httpx
    # 0.20+ does NOT follow redirects by default, so we must opt in explicitly.
    text = httpx.get(f"{BASE_URL}/metrics", timeout=5, follow_redirects=True).text
    for family in text_string_to_metric_families(text):         # iterate parsed metric families
        if family.name == name:                                 # find the family we want
            return family.samples                               # return all its samples
    return []                                                   # metric not found → empty list

def main():
    # ── requests_total counter ────────────────────────────────────────────────
    # The parser strips the _total suffix: Counter("requests_total") → family "requests".
    all_counter_samples = fetch_metric("requests")              # pull all counter samples
    samples = [s for s in all_counter_samples if s.name == "requests_total"]
    print("requests_total samples:")
    for s in samples:                                           # each s is a Sample namedtuple
        labels_str = ", ".join(f"{k}={v}" for k, v in s.labels.items())  # format label dict
        print(f"  {{{labels_str}}}  →  {s.value:.0f}")         # label set + current count

    # ── request_latency_seconds histogram ────────────────────────────────────
    # text_string_to_metric_families groups all histogram sub-metrics under the
    # base family name, so we fetch "request_latency_seconds" (not "_count").
    hist_samples = fetch_metric("request_latency_seconds")      # all histogram sub-samples
    hist_count = [s for s in hist_samples if s.name == "request_latency_seconds_count"]
    print("\nrequest_latency_seconds_count samples:")
    for s in hist_count:
        labels_str = ", ".join(f"{k}={v}" for k, v in s.labels.items())
        print(f"  {{{labels_str}}}  →  {s.value:.0f} observations")

    # ── inflight_requests gauge ───────────────────────────────────────────────
    gauge_samples = fetch_metric("inflight_requests")           # pull gauge samples
    print("\ninflight_requests:")
    for s in gauge_samples:
        print(f"  value = {s.value}")                          # should be 0.0 (no active requests)

if __name__ == "__main__":
    main()                                                      # entry point when run directly

Overwriting read_metrics.py


In [61]:
import subprocess, sys

result = subprocess.run(                                        # run read_metrics.py as a subprocess
    [sys.executable, "read_metrics.py"],                        # use the same Python as the notebook
    capture_output=True,                                        # capture stdout and stderr
    text=True,                                                  # decode bytes → str automatically
)
print(result.stdout)                                            # show parsed metric output
if result.stderr:                                               # print any errors if present
    print("STDERR:", result.stderr)

requests_total samples:
  {path=/weather, status=200}  →  5
  {path=/metrics, status=307}  →  3
  {path=/metrics/, status=200}  →  2

request_latency_seconds_count samples:
  {path=/weather}  →  5 observations
  {path=/metrics}  →  4 observations
  {path=/metrics/}  →  3 observations

inflight_requests:
  value = 1.0



In [ ]:
#import subprocess

#subprocess.run(["pkill", "-f", "uvicorn app:app"],              # send SIGTERM to uvicorn process
#               capture_output=True)
#print("Server stopped.")                                        # confirm shutdown

Server stopped.
